#### Day 1 - Define the Problem and Create Labels
  * Load merged_fire_data.csv
  * Define your target variable - will fire spread to an adjacent location?
  * Create binary labels from your data

In [114]:
import pandas as pd

In [115]:
merged_fire_df = pd.read_csv('../data/merged_fire_data.csv')

In [117]:
merged_fire_df.shape

(4071, 24)

In [ ]:
# 1. Sort your dataframe by datetime so detections are in chronological order

In [119]:
merged_fire_df = merged_fire_df.sort_values("datetime", ascending=True)

In [ ]:
merged_fire_df

,latitude,longitude,brightness,acq_date,acq_time,satellite,confidence,bright_t31,frp,daynight,...,relative_humidity_2m,wind_speed_10m,wind_direction_10m,elevation,slope,vegetation,wind_slope_alignment,vpd,fuel_dryness,fire_spread_risk
0,60.4007,-120.4775,312.8,2023-08-21,443,Terra,86,282.8,20.0,N,...,56,9.7,211,563.746343,0.701510,20,-9.337368,0.810324,0.289286,39.369722
1,59.9441,-119.5471,329.8,2023-08-21,443,Terra,100,282.9,40.4,N,...,56,9.7,211,580.000000,1.118034,20,-14.881471,0.810324,0.289286,126.746243
2,59.9373,-119.4737,313.1,2023-08-21,443,Terra,75,285.4,18.5,N,...,56,9.7,211,581.000000,0.707107,20,-9.411869,0.810324,0.289286,36.707555
3,59.9447,-119.3600,321.8,2023-08-21,443,Terra,100,283.4,27.0,N,...,56,9.7,211,584.000000,2.061553,20,-27.440076,0.810324,0.289286,156.191340
4,59.9283,-119.3926,335.7,2023-08-21,443,Terra,100,290.2,48.2,N,...,56,9.7,211,585.000000,0.707107,20,-9.411869,0.810324,0.289286,95.638061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4062,59.0070,-120.4357,336.7,2023-09-23,2108,Aqua,88,291.2,37.2,D,...,33,4.2,20,559.000000,1.118034,20,6.018593,1.435912,0.563636,98.456919
4061,59.0004,-120.4137,347.6,2023-09-23,2108,Aqua,95,290.1,57.7,D,...,33,4.2,20,527.000000,1.118034,20,6.018593,1.435912,0.563636,152.714092
4060,59.0144,-120.2332,326.8,2023-09-23,2108,Aqua,85,291.1,25.3,D,...,33,4.2,20,390.000000,0.500000,20,2.691597,1.435912,0.563636,29.946000
4064,59.0158,-120.4401,346.1,2023-09-23,2108,Aqua,82,291.4,52.7,D,...,33,4.2,20,568.000000,0.500000,20,2.691597,1.435912,0.563636,62.377636


In [124]:
time_stamps = merged_fire_df['datetime'].unique().tolist()

In [125]:
merged_fire_df['fire_spread'] = 0

In [126]:
for index, row in merged_fire_df.iterrows():
    latitude, longitude, datetime = row[['latitude', 'longitude', 'datetime']]
    time_index = time_stamps.index(datetime)
    next_index = time_index + 1
    if next_index >= len(time_stamps):
        continue
    next_time_stamp = time_stamps[next_index]
    next_detections = merged_fire_df[merged_fire_df['datetime'] == next_time_stamp]
    
    latitude_difference = abs(next_detections['latitude'] - latitude)
    longitude_difference = abs(next_detections['longitude'] - longitude)
    has_latitude_and_longitude_within_difference = ((latitude_difference <= 0.1) & (longitude_difference <= 0.1)).any()
    if has_latitude_and_longitude_within_difference:
        merged_fire_df.at[index, 'fire_spread'] = 1
    else:
        merged_fire_df.at[index, 'fire_spread'] = 0

In [127]:
merged_fire_df['fire_spread'].value_counts()

fire_spread
1    2334
0    1737
Name: count, dtype: int64

In [214]:
merged_fire_df.to_csv('../data/merged_fire_data.csv', index=False)

#### Day 2 - Prepare Features and Split Data
  * Select your feature columns
  * Handle any remaining missing values
  * Split into training and test sets

In [128]:
# 1. Select your feature columns - choose which columns to use as inputs to your model

In [129]:
feature_columns = ['latitude', 'longitude', 'brightness', 'frp', 'bright_t31', 'temperature_2m', 
                    'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m', 'elevation', 'slope', 'vegetation', 
                    'wind_slope_alignment', 'vpd', 'fuel_dryness', 'fire_spread_risk', 
                    'confidence']

In [130]:
# 2. Drop columns you don't need - things like acq_date, acq_time, satellite, daynight that aren't predictive features

In [131]:
# 3. Handle any remaining missing values
merged_fire_df[feature_columns].isnull().sum() # No missing values

latitude                0
longitude               0
brightness              0
frp                     0
bright_t31              0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
wind_direction_10m      0
elevation               0
slope                   0
vegetation              0
wind_slope_alignment    0
vpd                     0
fuel_dryness            0
fire_spread_risk        0
confidence              0
dtype: int64

In [132]:
# 4. Define X and y - X is your feature matrix, y is your fire_spread target column
X = merged_fire_df[feature_columns]

In [133]:
y = merged_fire_df['fire_spread']

In [134]:
from sklearn.model_selection import train_test_split

In [135]:
# 5. Split into training and test sets

In [136]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

In [137]:
X_train.shape, X_test.shape

((3256, 17), (815, 17))

#### Day 3 - Build a Baseline Model
  * Train a simple LogisticRegression first
  * Evaluate with precision, recall, and F1 score
  * Understand what the baseline tells you

In [138]:
from sklearn.linear_model import LogisticRegression

In [139]:
from sklearn.preprocessing import StandardScaler

In [140]:
standardScaler = StandardScaler()

In [141]:
X_train_scaled = standardScaler.fit_transform(X_train)

In [142]:
X_test_scaled = standardScaler.transform(X_test)

In [143]:
model = LogisticRegression(max_iter=1000)

In [144]:
model.fit(X_train_scaled, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [145]:
y_test_predictions = model.predict(X_test_scaled)

In [146]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [147]:
accuracy = accuracy_score(y_test, y_test_predictions)

In [148]:
accuracy

0.6760736196319018

In [149]:
cr = classification_report(y_test, y_test_predictions)
print(cr)

              precision    recall  f1-score   support

           0       0.61      0.58      0.60       335
           1       0.72      0.74      0.73       480

    accuracy                           0.68       815
   macro avg       0.66      0.66      0.66       815
weighted avg       0.67      0.68      0.67       815



In [150]:
cm = confusion_matrix(y_test, y_test_predictions)
print(cm)

[[194 141]
 [123 357]]


#### Day 4 - Build the Real Model
  * Train a Random Forest or Gradient Boosting classifier
  * Compare against your baseline
  * Tune basic hyperparameters

In [151]:
from sklearn.ensemble import RandomForestClassifier

In [152]:
rf = RandomForestClassifier(n_estimators=100, random_state=1234)

In [153]:
rf.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1234
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap boots

In [154]:
rf_test_predictions = rf.predict(X_test)

In [155]:
rf_accuracy = accuracy_score(y_test, rf_test_predictions)

In [156]:
f'{round(rf_accuracy * 100, 2)}%'

'90.55%'

#### Gradient Boosting Classifier

In [159]:
from sklearn.ensemble import GradientBoostingClassifier

In [160]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=1234)

In [161]:
gb.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",1234
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (

In [162]:
gb_test_predictions = gb.predict(X_test)

In [163]:
gb_accuracy = accuracy_score(y_test, gb_test_predictions)
f'{round(gb_accuracy * 100, 2)}%'

'86.63%'

In [164]:
gb_cr = classification_report(y_test, gb_test_predictions)
print(gb_cr)

              precision    recall  f1-score   support

           0       0.87      0.80      0.83       335
           1       0.87      0.91      0.89       480

    accuracy                           0.87       815
   macro avg       0.87      0.86      0.86       815
weighted avg       0.87      0.87      0.87       815



In [165]:
gb_cm = confusion_matrix(y_test, gb_test_predictions)
print(gb_cm)

[[267  68]
 [ 41 439]]


#### Perform Hyperparameter Tuning on Random Forest

#### Day 5 - Evaluate Honestly
  * Confusion matrix
  * Precision, recall, F1 by class
  * Feature importance - which variables matter most

In [166]:
rf_cr = classification_report(y_test, rf_test_predictions)
print(rf_cr)

              precision    recall  f1-score   support

           0       0.90      0.87      0.88       335
           1       0.91      0.93      0.92       480

    accuracy                           0.91       815
   macro avg       0.90      0.90      0.90       815
weighted avg       0.91      0.91      0.91       815



In [167]:
rf_cm = confusion_matrix(y_test, rf_test_predictions)
print(rf_cm)

[[292  43]
 [ 34 446]]


In [178]:
features = pd.DataFrame(rf.feature_importances_, index=X.columns)[0]

In [179]:
features.sort_values(ascending=False)

latitude                0.122080
longitude               0.102674
relative_humidity_2m    0.089735
temperature_2m          0.082604
vpd                     0.073148
fuel_dryness            0.070764
wind_direction_10m      0.063109
elevation               0.060303
frp                     0.057888
bright_t31              0.052925
wind_speed_10m          0.049515
brightness              0.047482
fire_spread_risk        0.044701
wind_slope_alignment    0.034642
confidence              0.032106
slope                   0.016323
vegetation              0.000000
Name: 0, dtype: float64

#### Day 6 - Hyperparameter Tuning
  * Run GridSearchCV on your Random Forest
  * Report best parameters and whether accuracy improved

In [204]:
from sklearn.model_selection import GridSearchCV

In [182]:
param_grid = [{
  'n_estimators': [100, 200, 300],
  'max_depth': [None, 10, 20, 30],
  'min_samples_split': [2, 5, 10]
}]

In [184]:
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)

In [185]:
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...om_state=1234)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'max_depth': [None, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, de

In [190]:
best_f1_score = float(grid_search.best_score_)

In [191]:
best_f1_score

0.8974225782317522

In [192]:
best_parameters = grid_search.best_params_
best_parameters

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

In [199]:
best_rf = grid_search.best_estimator_
best_rf_predictions = best_rf.predict(X_test)
best_rf_accuracy = round(accuracy_score(y_test, best_rf_predictions) * 100, 2)

In [200]:
f'{best_rf_accuracy}%'

'90.31%'

In [201]:
best_rf_cr = classification_report(y_test, best_rf_predictions)
print(best_rf_cr)

              precision    recall  f1-score   support

           0       0.90      0.87      0.88       335
           1       0.91      0.93      0.92       480

    accuracy                           0.90       815
   macro avg       0.90      0.90      0.90       815
weighted avg       0.90      0.90      0.90       815



In [202]:
best_rf_cm = confusion_matrix(y_test, best_rf_predictions)
print(best_rf_cm)

[[290  45]
 [ 34 446]]


In [211]:
best_rf_features = pd.DataFrame(best_rf.feature_importances_, index=X.columns)[0]

In [213]:
best_rf_features.sort_values(ascending=False)

latitude                0.122409
longitude               0.104173
relative_humidity_2m    0.089336
temperature_2m          0.083192
vpd                     0.070718
fuel_dryness            0.069868
wind_direction_10m      0.063593
elevation               0.060855
frp                     0.057410
bright_t31              0.052056
wind_speed_10m          0.049686
brightness              0.049183
fire_spread_risk        0.043833
wind_slope_alignment    0.035055
confidence              0.032852
slope                   0.015782
vegetation              0.000000
Name: 0, dtype: float64

#### Day 7 - Wrap Up Week 3
  * Save your trained model
  * Commit to GitHub

In [180]:
import joblib

In [205]:
joblib.dump(best_rf, '../models/wildfire_spread_model.pkl')

['../models/wildfire_spread_model.pkl']

### Summary
* In Week 3,
    * I created a target variable for my dataset based on the datetimes of each of the fire detections in my dataset. 
    * To build my target variable, I compared the latitude and longitude difference between the fires that occurred on the current datetime of the row we are currently iterating and those that happened on the next unique datetime in the merged dataset. If there were any fire detections that were within a 0.1 degree latitude and longitude difference. If so, that is a positive case of fire spread. If not, that is a negative case of fire spread

    * Model Selection
        * LogisticRegression yielded an accuracy of 67%, so I realized that I had to try the Random Forest Classifier
        * I also did try Gradient Boosting Classifier, but yielded an accuracy of 86%
        * I ended up going with Random Forest Classifier as it yielded slightly higher metrics across accuracy, precision, and recall when compared to GradientBoostingClassifier

    * Hyperparameter Tuning
        * Even though, RandomForestClassifier yielded high results. I still wanted to experiment with the model hyperparameters. The hyperparameters that I chose to experiment with were n_estimators, max_depth, and min_samples_split with the following configurations: [100, 200, 300], [None, 10, 20, 30], and [2, 5, 10] respectively.
        * I found that my results improved marginally from the baseline RandomForestClassifier
    
    * Best Model: Random Forest Classifier, Accuracy: 90%, Positive Class Precision: 91%, Positive Class Recall: 93%
        * Best Parameters: max_depth=None, min_samples_split=2, n_estimators=200

    * Latitude, Longitude, Relative Humidity, Temperature, Vapor Pressure Deficit were the top 5 most important features for the RandomForestClassifier when predicting fire detection spread